**Authors:** Pablo Rodríguez Elvira, Adrián Segura Onorato

# Design and implementation


In [27]:
import altair as alt
import pandas as pd
from pathlib import Path

input_file1 = Path("simpsons_character_totals.csv")
input_file2 = Path("simpsons_character_season.csv")
input_file3 = Path("simpsons_character_episode.csv")
input_file4 = Path("simpsons_lines.csv")

df_total = pd.read_csv(input_file1)
df_season = pd.read_csv(input_file2)
df_episode = pd.read_csv(input_file3)
df_lines = pd.read_csv(input_file4)

## 1. **Word count distribution by character** — Who speaks the most?

In [28]:
characters_ordered = sorted(df_total["character"].tolist())

tableau10 = [
    "#4e79a7", "#f28e2b", "#e15759", "#76b7b2",
    "#59a14f", "#edc948", "#b07aa1", "#ff9da7",
    "#9c755f", "#bab0ac"
]

color_scale = alt.Scale(domain=characters_ordered, range=tableau10)

selection = alt.selection_point(fields=["character"], bind="legend")

slider = alt.binding_range(min=1, max=25, step=1, name="Up to Season: ")
season_param = alt.param(value=25, bind=slider)

bar_chart = alt.Chart(df_season).mark_bar().encode(
    x=alt.X("sum(total_words):Q", title="Total Words"),
    y=alt.Y("character:N", sort="-x", title="Character"),
    color=alt.Color(
        "character:N",
        scale=color_scale,
        legend=alt.Legend(title="Character")
    ),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.15)),
    tooltip=[
        alt.Tooltip("character:N",        title="Character"),
        alt.Tooltip("sum(total_words):Q", title="Total Words", format=","),
    ]
).add_params(
    selection, season_param
).transform_filter(
    alt.datum.season <= season_param
).properties(
    title=alt.TitleParams("Total Word Count per Character", fontSize=18),
    width=500,
    height=300
)

bar_chart

alt.Chart(...)

### Design decisions


## 2. **Word Count Evolution Across Seasons** — How has character dialogue changed over time?

In [29]:
characters_ordered = sorted(df_total["character"].tolist())

tableau10 = [
    "#4e79a7", "#f28e2b", "#e15759", "#76b7b2",
    "#59a14f", "#edc948", "#b07aa1", "#ff9da7",
    "#9c755f", "#bab0ac"
]

color_scale = alt.Scale(domain=characters_ordered, range=tableau10)

y_max = int(df_season["total_words"].max()) + 500

characters = df_season["character"].unique().tolist()

character_dropdown = alt.binding_select(
    options=characters_ordered,
    name="Character: "
)
char_selection = alt.selection_point(
    fields=["character"],
    bind=character_dropdown,
    value=characters_ordered[0]
)

bar_season = alt.Chart(df_season).mark_bar().encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("total_words:Q", title="Total Words", scale=alt.Scale(domain=[0, y_max])),
    color=alt.Color(
        "character:N",
        scale=color_scale,
        legend=None
    ),
    tooltip=[
        alt.Tooltip("character:N", title="Character"),
        alt.Tooltip("season:O", title="Season"),
        alt.Tooltip("total_words:Q", title="Total Words", format=","),
    ]
).add_params(
    char_selection
).transform_filter(
    char_selection
).properties(
    title=alt.TitleParams("Word count per season", fontSize=18),
    width=600,
    height=300
)

bar_season

alt.Chart(...)

In [30]:
highlight = alt.selection_point(fields=["character"], bind="legend")

line_season = alt.Chart(df_season).mark_line(point=True).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("total_words:Q", title="Total Words"),
    color=alt.Color(
        "character:N",
        scale=alt.Scale(scheme="tableau10"),
        legend=alt.Legend(title="Character")
    ),
    opacity=alt.condition(highlight, alt.value(1.0), alt.value(0.1)),
    strokeWidth=alt.condition(highlight, alt.value(3), alt.value(1)),
    tooltip=[
        alt.Tooltip("character:N", title="Character"),
        alt.Tooltip("season:O", title="Season"),
        alt.Tooltip("total_words:Q", title="Total Words", format=","),
    ]
).add_params(
    highlight
).properties(
    title=alt.TitleParams("Word Count Evolution per Season", fontSize=18),
    width=600,
    height=300
)

line_season

alt.Chart(...)

### Design decisions


## 3. **Word Distribution Comparison by Season** — Comparing two characters throughout a season

In [31]:
seasons = sorted(df_episode["season"].unique().tolist())
characters_sorted = sorted(df_episode["character"].unique().tolist())

# Dropdowns
season_dropdown = alt.binding_select(options=seasons, name="Season: ")
char1_dropdown  = alt.binding_select(options=characters_sorted, name="Character 1: ")
char2_dropdown  = alt.binding_select(options=characters_sorted, name="Character 2: ")

season_sel = alt.selection_point(fields=["season"],    bind=season_dropdown, value=seasons[0])
char1_sel  = alt.selection_point(fields=["character"], bind=char1_dropdown,  value=characters_sorted[0])
char2_sel  = alt.selection_point(fields=["character"], bind=char2_dropdown,  value=characters_sorted[1])

# Filtramos el df a la season y a cada personaje por separado
base = alt.Chart(df_episode).transform_filter(season_sel)

bars_char1 = base.transform_filter(
    char1_sel
).mark_bar(opacity=0.85).encode(
    x=alt.X("number_in_season:O", title="Episode", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("total_words:Q", title="Total Words"),
    color=alt.Color("character:N", scale=color_scale, legend=alt.Legend(title="Character")),
    tooltip=[
        alt.Tooltip("character:N",       title="Character"),
        alt.Tooltip("title:N",           title="Episode Title"),
        alt.Tooltip("number_in_season:O",title="Episode"),
        alt.Tooltip("total_words:Q",     title="Total Words", format=","),
    ]
).add_params(char1_sel)

bars_char2 = base.transform_filter(
    char2_sel
).mark_bar(opacity=0.5).encode(
    x=alt.X("number_in_season:O", title="Episode"),
    y=alt.Y("total_words:Q", title="Total Words"),
    color=alt.Color("character:N", scale=color_scale, legend=alt.Legend(title="Character")),
    tooltip=[
        alt.Tooltip("character:N",       title="Character"),
        alt.Tooltip("title:N",           title="Episode Title"),
        alt.Tooltip("number_in_season:O",title="Episode"),
        alt.Tooltip("total_words:Q",     title="Total Words", format=","),
    ]
).add_params(char2_sel)

chart_q3 = (bars_char1 + bars_char2).add_params(
    season_sel
).properties(
    title=alt.TitleParams("Word Distribution Comparison by Season", fontSize=18),
    width=650,
    height=350
).resolve_scale(color="shared")

chart_q3

alt.LayerChart(...)

In [ ]:
# Max words per season across all characters — used to anchor the Y scale
season_max_df = (
    df_episode
    .groupby("season")["total_words"]
    .max()
    .reset_index()
    .rename(columns={"total_words": "season_max_words"})
)
season_max_df["season_max_words"] += 100

# Invisible layer: forces the Y axis max to the season-level maximum,
# so switching characters doesn't rescale the axis
y_anchor = alt.Chart(season_max_df).transform_filter(
    season_sel
).mark_point(opacity=0, size=0).encode(
    y=alt.Y("season_max_words:Q", title="Total Words", scale=alt.Scale(domainMin=0))
)

bars_char1_grouped = base.transform_filter(
    char1_sel
).mark_bar(width=15, xOffset=-9).encode(
    x=alt.X("number_in_season:O", title="Episode", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("total_words:Q"),
    color=alt.Color("character:N", scale=color_scale, legend=alt.Legend(title="Character")),
    tooltip=[
        alt.Tooltip("character:N",        title="Character"),
        alt.Tooltip("title:N",            title="Episode Title"),
        alt.Tooltip("number_in_season:O", title="Episode"),
        alt.Tooltip("total_words:Q",      title="Total Words", format=","),
    ]
)

bars_char2_grouped = base.transform_filter(
    char2_sel
).mark_bar(width=15, xOffset=9).encode(
    x=alt.X("number_in_season:O", title="Episode"),
    y=alt.Y("total_words:Q"),
    color=alt.Color("character:N", scale=color_scale, legend=alt.Legend(title="Character")),
    tooltip=[
        alt.Tooltip("character:N",        title="Character"),
        alt.Tooltip("title:N",            title="Episode Title"),
        alt.Tooltip("number_in_season:O", title="Episode"),
        alt.Tooltip("total_words:Q",      title="Total Words", format=","),
    ]
)

chart_q3_grouped = (y_anchor + bars_char1_grouped + bars_char2_grouped).add_params(
    char2_sel, char1_sel, season_sel
).properties(
    title=alt.TitleParams("Word Distribution Comparison by Season (Grouped)", fontSize=18),
    width=650,
    height=350
).resolve_scale(color="shared")

chart_q3_grouped

alt.LayerChart(...)

### Design decisions


## 4. **Word Distribution Comparison by Episode** — Comparing two characters in a single episode

### Design decisions


## 5. **Sentence Count Distribution by Character** — Who speaks the most sentences?

In [33]:
selection = alt.selection_point(fields=["character"], bind="legend")

bar_chart = alt.Chart(df_total).mark_bar().encode(
    x=alt.X("total_sentences:Q", title="Total Sentences"),
    y=alt.Y("character:N", sort="-x", title="Character"),
    color=alt.Color(
        "character:N",
        scale=alt.Scale(scheme="tableau10"), #Paleta para daltónicos
        legend=alt.Legend(title="Character")
    ),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.15)),
    tooltip=[
        alt.Tooltip("character:N", title="Character"),
        alt.Tooltip("total_sentences:Q", title="Total Sentences", format=",")
    ]
).add_params(
    selection
).properties(
    title=alt.TitleParams("Total Sentence Count per Character", fontSize=18),
    width=500,
    height=300
)

bar_chart

alt.Chart(...)

### Design decisions
